In [ ]:
# v45 -- Euclid P2: Filament Spin Anti-Correlation Test
# Pre-registered prediction: C_omega_x < 0 at r ~ T1 = 1660 Mpc
#
# KTF3: under phi(x,y,z) = (-x,y,z), angular momentum transforms:
#   L_x -> -L_x  (anti-correlated)
#   L_y -> +L_y  (correlated)
#   L_z -> +L_z  (correlated)
#
# Uses Euclid DR1 filament catalog (from DisPerSE or T-ReX algorithm)
# Geometry correction: redshift shuffling null (learned from v42b)
#
# PRE-REGISTERED: March 2026, before Euclid DR1 release.
# Cotting, S. (2026)

In [ ]:
# CELL 1 -- Imports
!pip install numpy matplotlib scipy astropy -q
import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from astropy.io import fits
import os, warnings
warnings.filterwarnings('ignore')

cosmo = FlatLambdaCDM(H0=67.4, Om0=0.315)

print('v45 -- Euclid P2: Filament Spin Anti-Correlation')
print('PRE-REGISTERED March 2026 -- Cotting, S.')
print('='*60)
print()
print('KTF3 Prediction P2:')
print('  C_omega_x < 0 at r ~ T1 = 1660 Mpc  (anti-correlated)')
print('  C_omega_y > 0 at r ~ T1 = 1660 Mpc  (correlated)')
print('  C_omega_z > 0 at r ~ T1 = 1660 Mpc  (correlated)')
print()
print('Method: geometry-corrected (redshift shuffling null)')
print('  Learned from v42b: random orientation shuffling is insufficient')
print('  Euclid uniform sky coverage reduces geometry bias vs SDSS')

In [ ]:
# CELL 2 -- Load Euclid filament catalog
# Euclid DR1 will include filament catalogs from:
# - DisPerSE (Sousbie 2011)
# - T-ReX (Bonnaire et al. 2022)
# Expected format: FITS with columns RA1,DEC1,Z1,RA2,DEC2,Z2 (endpoints)

FILAMENT_FILE = 'euclid_dr1_filaments.fits'  # << REPLACE WITH ACTUAL FILE

if not os.path.exists(FILAMENT_FILE):
    print('Euclid filament catalog not yet available.')
    print('Generating synthetic Euclid-like filaments...')
    N_fil = 50000
    rng = np.random.default_rng(42)
    ra1  = rng.uniform(0, 360, N_fil)
    dec1 = np.degrees(np.arcsin(rng.uniform(-0.5, 0.9, N_fil)))
    z1   = rng.uniform(0.1, 2.0, N_fil)
    ra2  = ra1  + rng.normal(0, 0.1, N_fil)
    dec2 = dec1 + rng.normal(0, 0.1, N_fil)
    z2   = z1   + rng.uniform(0.001, 0.05, N_fil)
    USE_SYNTHETIC = True
    print(f'Synthetic filaments: {N_fil}')
else:
    hdul = fits.open(FILAMENT_FILE)
    data = hdul[1].data
    ra1, dec1, z1 = data['RA1'], data['DEC1'], data['Z1']
    ra2, dec2, z2 = data['RA2'], data['DEC2'], data['Z2']
    USE_SYNTHETIC = False
    print(f'Filaments loaded: {len(ra1)}')

# Convert to comoving Cartesian
def sky_to_cart(ra, dec, z):
    chi = cosmo.comoving_distance(z).value
    r, d = np.radians(ra), np.radians(dec)
    return np.stack([chi*np.cos(d)*np.cos(r),
                     chi*np.cos(d)*np.sin(r),
                     chi*np.sin(d)], axis=1)

pos1 = sky_to_cart(ra1, dec1, z1)
pos2 = sky_to_cart(ra2, dec2, z2)
midpoints    = (pos1 + pos2) / 2
orientations = pos2 - pos1
norms = np.linalg.norm(orientations, axis=1, keepdims=True)
norms = np.where(norms > 0, norms, 1.0)
orientations /= norms
ra_mid  = (ra1 + ra2) / 2
dec_mid = (dec1 + dec2) / 2
z_mid   = (z1 + z2) / 2
N_fil   = len(midpoints)
print(f'Filaments processed: {N_fil}')

In [ ]:
# CELL 3 -- Sample pairs and compute spin correlations
T1 = 1660  # Mpc
N_PAIRS = 600000
N_BINS  = 24
bins = np.linspace(100, 2500, N_BINS+1)
bin_centers = 0.5*(bins[:-1]+bins[1:])
T1_bin = np.argmin(np.abs(bin_centers - T1))

rng = np.random.default_rng(42)
idx1 = rng.integers(0, N_fil, N_PAIRS)
idx2 = rng.integers(0, N_fil, N_PAIRS)
keep = idx1 != idx2
idx1, idx2 = idx1[keep], idx2[keep]

dr = midpoints[idx2] - midpoints[idx1]
r  = np.linalg.norm(dr, axis=1)
n1, n2 = orientations[idx1], orientations[idx2]
px, py, pz = n1[:,0]*n2[:,0], n1[:,1]*n2[:,1], n1[:,2]*n2[:,2]

def bin_corr(prod, r, bins):
    C = np.zeros(len(bins)-1)
    for b in range(len(bins)-1):
        s = (r >= bins[b]) & (r < bins[b+1])
        if s.sum() > 10: C[b] = prod[s].mean()
    return C

Cx_obs = bin_corr(px, r, bins)
Cy_obs = bin_corr(py, r, bins)
Cz_obs = bin_corr(pz, r, bins)
print(f'At T1={T1} Mpc: Cx={Cx_obs[T1_bin]:.4f}, Cy={Cy_obs[T1_bin]:.4f}')

In [ ]:
# CELL 4 -- Geometry-corrected null (redshift shuffling)
# KEY LESSON from v42b: shuffle redshifts, not orientations
N_MC = 300
print(f'Geometry null: redshift shuffling (N={N_MC})...')
mc_Cx = np.zeros((N_MC, N_BINS))
mc_Cy = np.zeros((N_MC, N_BINS))
rng_mc = np.random.default_rng(99)

for i in range(N_MC):
    z_shuf = rng_mc.permutation(z_mid)
    chi_s  = cosmo.comoving_distance(z_shuf).value
    rr, dd = np.radians(ra_mid), np.radians(dec_mid)
    mid_s  = np.stack([chi_s*np.cos(dd)*np.cos(rr),
                       chi_s*np.cos(dd)*np.sin(rr),
                       chi_s*np.sin(dd)], axis=1)
    dr_s = mid_s[idx2] - mid_s[idx1]
    r_s  = np.linalg.norm(dr_s, axis=1)
    mc_Cx[i] = bin_corr(px, r_s, bins)
    mc_Cy[i] = bin_corr(py, r_s, bins)
    if (i+1) % 75 == 0: print(f'  MC {i+1}/{N_MC}')

resid_x = Cx_obs - mc_Cx.mean(axis=0)
resid_y = Cy_obs - mc_Cy.mean(axis=0)
sigma_x = resid_x / (mc_Cx.std(axis=0) + 1e-10)
sigma_y = resid_y / (mc_Cy.std(axis=0) + 1e-10)

print(f'\nAt T1={T1} Mpc:')
print(f'  sigma_x = {sigma_x[T1_bin]:.2f}  (KTF3: < 0)')
print(f'  sigma_y = {sigma_y[T1_bin]:.2f}  (KTF3: > 0)')

In [ ]:
# CELL 5 -- Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
for ax in axes:
    ax.set_facecolor('#0d1117'); ax.tick_params(colors='white'); ax.spines[:].set_color('#444')

ax = axes[0]
ax.plot(bin_centers, resid_x, color='#EF5350', lw=2, label='Residual C_x')
ax.plot(bin_centers, resid_y, color='#42A5F5', lw=2, label='Residual C_y')
ax.axhline(0, color='white', lw=0.5, alpha=0.4)
ax.axvline(T1, color='yellow', lw=2, ls='--', label=f'T1={T1} Mpc')
ax.set_xlabel('r [Mpc]', color='white'); ax.set_ylabel('Residual C_omega', color='white')
ax.set_title('P2: Spin Residual\n(geometry corrected)', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')

ax = axes[1]
ax.plot(bin_centers, sigma_x, color='#EF5350', lw=2.5, label=f'sigma_x (T1:{sigma_x[T1_bin]:.2f})')
ax.plot(bin_centers, sigma_y, color='#42A5F5', lw=2.5, label=f'sigma_y (T1:{sigma_y[T1_bin]:.2f})')
ax.axhline(0,  color='white',  lw=0.5, alpha=0.4)
ax.axhline(2,  color='orange', lw=1.5, ls='--', alpha=0.7, label='2sigma')
ax.axhline(-2, color='orange', lw=1.5, ls='--', alpha=0.7)
ax.axvline(T1, color='yellow', lw=2, ls='--', label=f'T1={T1}')
ax.set_xlabel('r [Mpc]', color='white'); ax.set_ylabel('Sigma', color='white')
ax.set_title('P2: Spin Significance\n(KEY RESULT)', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8)

plt.suptitle('v45: Euclid P2 Spin Anti-Correlation -- Cotting (2026)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('v45_euclid_p2.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# CELL 6 -- Summary
sx, sy = sigma_x[T1_bin], sigma_y[T1_bin]
print('\n' + '='*60)
print('RESULT v45: EUCLID P2 SPIN ANTI-CORRELATION')
print('Cotting, S. (2026) -- PRE-REGISTERED March 2026')
print('='*60)
print(f'  T1 = {T1} Mpc, geometry-corrected null')
print(f'  sigma_x = {sx:.2f}  (KTF3: < 0)')
print(f'  sigma_y = {sy:.2f}  (KTF3: > 0)')
print()
if sx < -2 and sy > 0:
    verdict = 'CONFIRMED -- Anti-correlation at T1. KTF3 consistent.'
elif sx < 0 and sy > 0:
    verdict = 'MARGINAL -- Correct sign pattern, not significant.'
elif sx > 2:
    verdict = 'FALSIFIED -- Wrong sign (sigma_x > 0).'
else:
    verdict = 'NULL -- No significant signal at T1.'
print(f'  Verdict: {verdict}')
print('='*60)